## Search algorithms within Optuna

In this notebook, I will demo how to select the search algorithm with Optuna. We will compare the use of:

- Grid Search 
- Randomized search
- Tree-structured Parzen Estimators
- CMA-ES


We can select the search algorithm from the [optuna.study.create_study()](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.create_study.html#optuna.study.create_study) class.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier

import optuna

g:\broswer download\udemy_courses\hyperparameter_optimization_for_ml\hyperparameter-optimization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load dataset

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
y = y.map({0:1, 1:0})

X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# the target:
# percentage of benign (0) and malign tumors (1)

y.value_counts() / len(y)

target
0    0.627417
1    0.372583
Name: count, dtype: float64

In [4]:
# split dataset into a train and test set

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0)

X_train.shape, X_test.shape

((398, 30), (171, 30))

## Define the objective function

This is the hyperparameter response space, the function we want to minimize.

In [5]:
# the objective function takes the hyperparameter space
# as input

def objective(trial):

    rf_n_estimators = trial.suggest_int("rf_n_estimators", 100, 1000)
    rf_criterion = trial.suggest_categorical("rf_criterion", ['gini', 'entropy'])
    rf_max_depth = trial.suggest_int("rf_max_depth", 1, 4)
    rf_min_samples_split = trial.suggest_float("rf_min_samples_split", 0.01, 1)
    
    model = RandomForestClassifier(
        n_estimators=rf_n_estimators,
        criterion=rf_criterion,
        max_depth=rf_max_depth,
        min_samples_split=rf_min_samples_split,
    )

    score = cross_val_score(model, X_train, y_train, cv=3)
    accuracy = score.mean()
    return accuracy

## Randomized Search

[RandomSampler()](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.RandomSampler.html)

In [6]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.RandomSampler(),
)


study.optimize(objective, n_trials=20)

[I 2026-08-27 00:03:05,846] A new study created in memory with name: no-name-97582cbe-c40a-4581-8382-041ee8002f7a
[I 2026-08-27 00:03:07,030] Trial 0 finished with value: 0.6256360598465861 and parameters: {'rf_n_estimators': 361, 'rf_criterion': 'gini', 'rf_max_depth': 4, 'rf_min_samples_split': 0.9123810249149172}. Best is trial 0 with value: 0.6256360598465861.
[I 2026-08-27 00:03:08,667] Trial 1 finished with value: 0.6256360598465861 and parameters: {'rf_n_estimators': 509, 'rf_criterion': 'gini', 'rf_max_depth': 4, 'rf_min_samples_split': 0.8573340570100346}. Best is trial 0 with value: 0.6256360598465861.
[I 2026-08-27 00:03:09,741] Trial 2 finished with value: 0.912128806865649 and parameters: {'rf_n_estimators': 259, 'rf_criterion': 'gini', 'rf_max_depth': 2, 'rf_min_samples_split': 0.6262504362238779}. Best is trial 2 with value: 0.912128806865649.
[I 2026-08-27 00:03:13,801] Trial 3 finished with value: 0.9171223513328776 and parameters: {'rf_n_estimators': 975, 'rf_criterio

In [7]:
study.best_params

{'rf_n_estimators': 741,
 'rf_criterion': 'entropy',
 'rf_max_depth': 2,
 'rf_min_samples_split': 0.1300668389982579}

In [8]:
study.best_value

0.9397357028935976

In [9]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_rf_criterion,params_rf_max_depth,params_rf_min_samples_split,params_rf_n_estimators,state
0,0,0.625636,2026-08-27 00:03:05.848904,2026-08-27 00:03:07.030183,0 days 00:00:01.181279,gini,4,0.912381,361,COMPLETE
1,1,0.625636,2026-08-27 00:03:07.031687,2026-08-27 00:03:08.667768,0 days 00:00:01.636081,gini,4,0.857334,509,COMPLETE
2,2,0.912129,2026-08-27 00:03:08.668768,2026-08-27 00:03:09.741625,0 days 00:00:01.072857,gini,2,0.626250,259,COMPLETE
3,3,0.917122,2026-08-27 00:03:09.743678,2026-08-27 00:03:13.801827,0 days 00:00:04.058149,gini,1,0.171422,975,COMPLETE
4,4,0.625636,2026-08-27 00:03:13.802348,2026-08-27 00:03:16.804373,0 days 00:00:03.002025,entropy,2,0.769907,920,COMPLETE
5,5,0.939736,2026-08-27 00:03:16.805372,2026-08-27 00:03:20.119413,0 days 00:00:03.314041,entropy,2,0.130067,741,COMPLETE
6,6,0.625636,2026-08-27 00:03:20.120415,2026-08-27 00:03:21.771334,0 days 00:00:01.650919,gini,2,0.859214,436,COMPLETE
7,7,0.625636,2026-08-27 00:03:21.773218,2026-08-27 00:03:23.885165,0 days 00:00:02.111947,entropy,4,0.832130,606,COMPLETE
8,8,0.917122,2026-08-27 00:03:23.886164,2026-08-27 00:03:25.082370,0 days 00:00:01.196206,entropy,2,0.511433,300,COMPLETE
9,9,0.625636,2026-08-27 00:03:25.084889,2026-08-27 00:03:25.695136,0 days 00:00:00.610247,entropy,3,0.938783,110,COMPLETE


## TPE

TPESampler is the default. You'll probably remember that for TPE, we first sample a few hyperparameter combinations at random, and then we start the sequential search, based on TPE with a certain kernel, and an acquisition function.

All these parameters are determined by default, but you can change them. More details in the [TPESampler()](https://optuna.readthedocs.io/en/stable/reference/samplers/generated/optuna.samplers.TPESampler.html) documentation.

In [10]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        # the number of initial points to sample at random before doing the sequential search
        n_startup_trials=10
    ),
)

study.optimize(objective, n_trials=20)

[I 2026-08-27 00:03:44,687] A new study created in memory with name: no-name-1f06efa6-a40d-45e6-a955-7add4a096998
[I 2026-08-27 00:03:46,385] Trial 0 finished with value: 0.9271853877117034 and parameters: {'rf_n_estimators': 377, 'rf_criterion': 'entropy', 'rf_max_depth': 2, 'rf_min_samples_split': 0.45434547315577234}. Best is trial 0 with value: 0.9271853877117034.
[I 2026-08-27 00:03:47,008] Trial 1 finished with value: 0.9171413381939697 and parameters: {'rf_n_estimators': 152, 'rf_criterion': 'entropy', 'rf_max_depth': 1, 'rf_min_samples_split': 0.4000176546706351}. Best is trial 0 with value: 0.9271853877117034.
[I 2026-08-27 00:03:49,314] Trial 2 finished with value: 0.9447482342219184 and parameters: {'rf_n_estimators': 501, 'rf_criterion': 'entropy', 'rf_max_depth': 2, 'rf_min_samples_split': 0.08840676752084796}. Best is trial 2 with value: 0.9447482342219184.
[I 2026-08-27 00:03:52,072] Trial 3 finished with value: 0.6256360598465861 and parameters: {'rf_n_estimators': 617,

In [11]:
study.best_params

{'rf_n_estimators': 501,
 'rf_criterion': 'entropy',
 'rf_max_depth': 2,
 'rf_min_samples_split': 0.08840676752084796}

In [12]:
study.best_value

0.9447482342219184

## CMA-ES

CmaEsSampler

In [13]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.CmaEsSampler(),
)

study.optimize(objective, n_trials=5)

[I 2026-08-27 00:04:47,309] A new study created in memory with name: no-name-44658c20-d9df-4c2f-8d29-b42b98612eba
[I 2026-08-27 00:04:48,160] Trial 0 finished with value: 0.9171223513328776 and parameters: {'rf_n_estimators': 158, 'rf_criterion': 'entropy', 'rf_max_depth': 2, 'rf_min_samples_split': 0.48611185505070753}. Best is trial 0 with value: 0.9171223513328776.
[W 2026-08-27 00:04:48,312] The parameter `rf_criterion` in Trial#1 is sampled independently using `RandomSampler` instead of `CmaEsSampler`, potentially degrading the optimization performance. This fallback happened because dynamic search space and `CategoricalDistribution` are not supported by `CmaEsSampler`. You can suppress this warning by setting `warn_independent_sampling` to `False` in the constructor of `CmaEsSampler` if this independent sampling is intended behavior.
[I 2026-08-27 00:04:50,659] Trial 1 finished with value: 0.6256360598465861 and parameters: {'rf_n_estimators': 527, 'rf_criterion': 'gini', 'rf_max

In [14]:
study.best_params

{'rf_n_estimators': 322,
 'rf_criterion': 'gini',
 'rf_max_depth': 2,
 'rf_min_samples_split': 0.4255474818375359}

In [15]:
study.best_value

0.9346851978430926

## Grid Search

GridSampler()

We are probably not going to perform GridSearch with Optuna, but in case you wanted to, you need to add a variable with the space, with the exact values that you want to be tested.

In [16]:
search_space = {
    "rf_n_estimators": [100, 500, 1000],
    "rf_criterion": ['gini', 'entropy'],
    "rf_max_depth": [1, 2, 3],
    "rf_min_samples_split": [0.1, 1.0]
}


In [17]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.GridSampler(search_space),
)

study.optimize(objective)

[I 2026-08-27 00:04:58,920] A new study created in memory with name: no-name-5fc8e449-08b2-4d13-983b-a2d64205008f
[I 2026-08-27 00:05:01,511] Trial 0 finished with value: 0.9372294372294373 and parameters: {'rf_n_estimators': 500, 'rf_criterion': 'entropy', 'rf_max_depth': 3, 'rf_min_samples_split': 0.1}. Best is trial 0 with value: 0.9372294372294373.
[I 2026-08-27 00:05:05,548] Trial 1 finished with value: 0.9146160856687172 and parameters: {'rf_n_estimators': 1000, 'rf_criterion': 'entropy', 'rf_max_depth': 1, 'rf_min_samples_split': 0.1}. Best is trial 0 with value: 0.9372294372294373.
[I 2026-08-27 00:05:07,560] Trial 2 finished with value: 0.6256360598465861 and parameters: {'rf_n_estimators': 500, 'rf_criterion': 'gini', 'rf_max_depth': 3, 'rf_min_samples_split': 1.0}. Best is trial 0 with value: 0.9372294372294373.
[I 2026-08-27 00:05:08,052] Trial 3 finished with value: 0.942222981696666 and parameters: {'rf_n_estimators': 100, 'rf_criterion': 'entropy', 'rf_max_depth': 3, 'rf

In [18]:
study.best_params

{'rf_n_estimators': 1000,
 'rf_criterion': 'gini',
 'rf_max_depth': 2,
 'rf_min_samples_split': 0.1}

In [19]:
study.best_value

0.942241968557758